In [15]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks, correlate
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, GRU, Dense, Dropout, Bidirectional, Attention, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

In [16]:
# =========================
# STEP 1: LOAD DATA
# =========================
data = pd.read_csv('samples3.csv')
data.columns = data.columns.str.strip().str.replace("'", "").str.replace(" ", "")

def clean_signal(signal):
    cleaned = []
    for x in signal:
        try:
            cleaned.append(float(str(x).replace("mV", "").strip()))
        except:
            continue
    return np.array(cleaned)

pleth = clean_signal(data['PLETH'].values[:15000])
abp   = clean_signal(data['ABP'].values[:15000])


C:\Users\abhis\AppData\Local\Temp\ipykernel_6340\4124931572.py:4: DtypeWarning: Columns (0,1,2,3,4,5) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('samples3.csv')


In [17]:
# =========================
# STEP 2: WEIGHTED MOVING AVERAGE (4th ORDER)
# =========================
def weighted_moving_average(signal, window=5):
    weights = np.arange(1, window+1)
    return np.convolve(signal, weights/weights.sum(), mode='same')

pleth_filtered = weighted_moving_average(pleth, 5)

In [18]:
# =========================
# STEP 3: LATCH REMOVAL + ALIGN LENGTH
# =========================
min_len = min(len(pleth_filtered), len(abp))
pleth_filtered = pleth_filtered[:min_len]
abp = abp[:min_len]

In [19]:
# =========================
# STEP 4: SYNCHRONIZATION
# =========================
corr = correlate(pleth_filtered, abp, mode='full')
delay = np.argmax(corr) - len(abp) + 1

if delay > 0:
    pleth_sync = pleth_filtered[delay:]
    abp_sync   = abp[:-delay]
else:
    pleth_sync = pleth_filtered[:delay]
    abp_sync   = abp[-delay:]


In [20]:
# =========================
# STEP 5: ADVANCED FEATURE EXTRACTION
# =========================
def extract_features(signal):
    peaks, _ = find_peaks(signal, distance=30)
    features = []

    for i in range(1, len(peaks)-1):
        systolic = signal[peaks[i]]
        notch = np.min(signal[peaks[i]:peaks[i+1]])

        # Additional features
        time_diff = peaks[i+1] - peaks[i]
        amplitude = systolic - notch
        area = np.trapz(signal[peaks[i]:peaks[i+1]])

        features.append([systolic, notch, time_diff, amplitude, area])

    return np.array(features)

features = extract_features(pleth_sync)
abp_sync = abp_sync[:len(features)]

C:\Users\abhis\AppData\Local\Temp\ipykernel_6340\504047936.py:15: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  area = np.trapz(signal[peaks[i]:peaks[i+1]])


In [21]:
# =========================
# STEP 6: NORMALIZATION (BETTER)
# =========================
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_norm = scaler_X.fit_transform(features)
y_norm = scaler_y.fit_transform(abp_sync.reshape(-1,1))

In [22]:
# =========================
# STEP 7: SEQUENCES (LONGER)
# =========================
def create_sequences(X, y, seq_len=50):
    Xs, ys = [], []
    for i in range(len(X)-seq_len):
        Xs.append(X[i:i+seq_len])
        ys.append(y[i+seq_len])
    return np.array(Xs), np.array(ys)

X, y = create_sequences(X_norm, y_norm, 50)


In [23]:
# =========================
# STEP 8: REMOVE OUTLIERS
# =========================
mask = np.abs(y - np.mean(y)) < 3*np.std(y)
X = X[mask.flatten()]
y = y[mask.flatten()]

In [24]:
# =========================
# STEP 9: TRAIN TEST SPLIT
# =========================
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

In [25]:
# =========================
# STEP 10: Bi-DIRECTIONAL LSTM MODEL
# =========================
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional

model = Sequential([
    Bidirectional(LSTM(128, return_sequences=True), input_shape=(X.shape[1], X.shape[2])),
    Bidirectional(LSTM(64)),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1)
])

model.compile(optimizer=Adam(learning_rate=0.0005), loss='mse')
model.summary()

c:\Users\abhis\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\rnn\bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional_2 (Bidirectional) │ (None, 50, 256)        │       137,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ (None, 128)            │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 309,889 (1.18 MB)

 Trainable params: 309,889 (1.18 MB)

 Non-trainable params: 0 (0.00 B)

In [26]:
# =========================
# STEP 11: TRAINING
# =========================
early_stop = EarlyStopping(patience=10, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_data=(X_test, y_test),
    callbacks=[early_stop]
)

Epoch 1/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 4s 604ms/step - loss: 0.8038 - val_loss: 1.1042
Epoch 2/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step - loss: 0.6686 - val_loss: 0.9903
Epoch 3/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step - loss: 0.5870 - val_loss: 0.8713
Epoch 4/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 111ms/step - loss: 0.4590 - val_loss: 0.7242
Epoch 5/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 112ms/step - loss: 0.3337 - val_loss: 0.5307
Epoch 6/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 111ms/step - loss: 0.2168 - val_loss: 0.2779
Epoch 7/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 125ms/step - loss: 0.0918 - val_loss: 0.0484
Epoch 8/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 153ms/step - loss: 0.0784 - val_loss: 0.0304
Epoch 9/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 198ms/step - loss: 0.1025 - val_loss: 0.0265
Epoch 10/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 198ms/step - loss: 0.0706 - val_loss: 0.0219
Epoch 11/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 206ms/step - loss: 0.0702 - val_loss: 0.0489
Epoch 12/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 198ms/step - loss: 0.0

In [31]:
# =========================
# STEP 13: METRICS (FINAL CLEAN VERSION)
# =========================

# --- Real scale (mmHg) ---
mse = mean_squared_error(y_test_inv, y_pred_inv)
mae = mean_absolute_error(y_test_inv, y_pred_inv)
sd  = np.std(y_test_inv - y_pred_inv)

# --- Normalized (0–1 scale using model values) ---
mse_norm = mean_squared_error(y_test, y_pred)
mae_norm = mean_absolute_error(y_test, y_pred)
sd_norm  = np.std(y_test - y_pred)

# =========================
# PRINT RESULTS
# =========================
print("\n===== FINAL RESULTS =====")

print("\n--- Real Scale (mmHg) ---")
print(f"MSE : {mse:.4f}")
print(f"MAE : {mae:.4f}")
print(f"SD  : {sd:.4f}")

print("\n--- Normalized (0 to 1) ---")
print(f"MSE (Normalized): {mse_norm:.6f}")
print(f"MAE (Normalized): {mae_norm:.6f}")
print(f"SD  (Normalized): {sd_norm:.6f}")


===== FINAL RESULTS =====

--- Real Scale (mmHg) ---
MSE : 29.0926
MAE : 4.6605
SD  : 5.3938

--- Normalized (0 to 1) ---
MSE (Normalized): 0.011854
MAE (Normalized): 0.094073
SD  (Normalized): 0.108875
